# 01 — Geography

Builds the master India geography file with LGD codes and district boundaries.

**Sources**
- LGD master CSVs from data.gov.in (states, districts, sub-districts) — October 2024
- District GeoJSON: datta07/INDIAN-SHAPEFILES — updated 2024
- Sub-district GeoJSON: datta07/INDIAN-SHAPEFILES — used to reconstruct 9 missing Karnataka districts

**Outputs**
- `data/processed/india_master_geography.csv` — state/district/sub-district hierarchy with LGD codes
- `data/processed/india_districts_with_lgd.geojson` — district boundaries joined to LGD codes

**Coverage**: 783/~800 districts (~98%). The ~17 missing are districts created after our data sources were compiled.

## 1. Imports and paths

In [ ]:
import pandas as pd
import geopandas as gpd
import re
from pathlib import Path

RAW       = Path("../data/raw")
PROCESSED = Path("../data/processed")
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print("Paths ready")

## 2. Load LGD master files

Official government codes used across all Indian government datasets.

In [ ]:
states       = pd.read_csv(RAW / "lgd_states.csv")
districts    = pd.read_csv(RAW / "lgd_districts.csv")
subdistricts = pd.read_csv(RAW / "lgd_subdistricts.csv")

print(f"States:        {states.shape}")
print(f"Districts:     {districts.shape}")
print(f"Sub-districts: {subdistricts.shape}")

## 3. Build master geography hierarchy

State → district → sub-district, with `state_or_ut` merged from states file.

In [ ]:
master = subdistricts[[
    'state_code', 'state_name_english', 'state_census2011_code',
    'district_code', 'district_name_english', 'district_census2011_code',
    'subdistrict_code', 'subdistrict_name_english', 'subdistrict_census2011_code',
]].copy()

master = master.merge(states[['state_code', 'state_or_ut']], on='state_code', how='left')

print(f"Shape: {master.shape}")
print(f"States: {master['state_name_english'].nunique()}")
print(f"Districts: {master['district_name_english'].nunique()}")
print(f"Sub-districts: {master['subdistrict_name_english'].nunique()}")
print(f"States vs UTs: {master.drop_duplicates('state_code')['state_or_ut'].value_counts().to_dict()}")

master.to_csv(PROCESSED / "india_master_geography.csv", index=False)
print("\nSaved: india_master_geography.csv")

## 4. Load GeoJSON files

District boundaries + sub-district boundaries (used to reconstruct missing Karnataka districts).

In [ ]:
gdf     = gpd.read_file(RAW / "india_districts.geojson")
gdf_sub = gpd.read_file(RAW / "india_subdistricts.geojson")

print(f"District GeoJSON:     {gdf.shape}  |  CRS: {gdf.crs}")
print(f"Sub-district GeoJSON: {gdf_sub.shape}")
print(f"\nDistrict columns: {gdf.columns.tolist()}")

## 5. Name cleaning functions

Three issues to handle:
1. **Encoding corruption** in the GeoJSON: `>` = A, `|` = I, `#` = U, `@` = H, `\` = I
2. **Normalisation**: uppercase, strip, preserve Gramin/Rural/Urban variants, remove hyphens
3. **Manual mapping**: ~80 spelling variants keyed on (district_norm, state_norm) to avoid cross-state false matches

In [ ]:
def fix_encoding(name):
    if not isinstance(name, str):
        return name
    return (name
        .replace('>', 'A').replace('|', 'I').replace('#', 'U')
        .replace('@', 'H').replace('\\', 'I')
    )

def normalise_name(name):
    """Uppercase, preserve Gramin/Rural/Urban variants, remove other parentheses, strip hyphens."""
    if not isinstance(name, str):
        return name
    name = name.upper().strip()
    name = re.sub(r'\(GRAMIN\)', ' GRAMIN', name)
    name = re.sub(r'\(RURAL\)',  ' RURAL',  name)
    name = re.sub(r'\(URBAN\)',  ' URBAN',  name)
    name = re.sub(r'\(.*?\)', '', name)
    name = name.replace('-', ' ').replace('&', 'AND')
    name = re.sub(r'\s+', ' ', name).strip()
    return name

def normalise_state(name):
    if not isinstance(name, str):
        return name
    name = name.upper().strip().replace('&', 'AND')
    return re.sub(r'\s+', ' ', name).strip()

# State name overrides
STATE_MAP = {
    'THE DADRA AND NAGAR HAVELI AND DAMAN AND DIU': 'DADRA AND NAGAR HAVELI AND DAMAN AND DIU',
    'GUJARAT AND DNH AND DD ISLANDS':               'DADRA AND NAGAR HAVELI AND DAMAN AND DIU',
}

# Manual district map: (geojson_district_norm, state_norm) -> lgd_district_norm
# None = genuinely unmappable (Pakistan-administered, new districts not yet in LGD)
DISTRICT_MANUAL_MAP = {
    ('AHMADABAD',                    'GUJARAT'):              'AHMEDABAD',
    ('ALIPUR DUAR',                  'WEST BENGAL'):          'ALIPURDUAR',
    ('ANOOPGARH',                    'RAJASTHAN'):            'ANUPGARH',
    ('BADGAM',                       'JAMMU AND KASHMIR'):    'BUDGAM',
    ('BALASORE',                     'ODISHA'):               'BALESHWAR',
    ('BALODA BAZAR',                 'CHHATTISGARH'):         'BALODABAZAR BHATAPARA',
    ('BALRAMPUR',                    'CHHATTISGARH'):         'BALRAMPUR RAMANUJGANJ',
    ('BANDIPURA',                    'JAMMU AND KASHMIR'):    'BANDIPORA',
    ('BARAMULA',                     'JAMMU AND KASHMIR'):    'BARAMULLA',
    ('BIRBHHM',                      'WEST BENGAL'):          'BIRBHUM',
    ('CHITTAURGARH',                 'RAJASTHAN'):            'CHITTORGARH',
    ('DAKSHIN BASTAR DANTEWARA',     'CHHATTISGARH'):         'DAKSHIN BASTAR DANTEWADA',
    ('DARANG',                       'ASSAM'):                'DARRANG',
    ('DARJILING',                    'WEST BENGAL'):          'DARJEELING',
    ('DAVANAGERE',                   'KARNATAKA'):            'DAVANGERE',
    ('DEHRADHN',                     'UTTARAKHAND'):          'DEHRADUN',
    ('DHAULPUR',                     'RAJASTHAN'):            'DHOLPUR',
    ('DR.B.R.AMBEDKAR KONASEEMA',    'ANDHRA PRADESH'):       'DR. B.R. AMBEDKAR KONASEEMA',
    ('EAST SINGHBHUM',               'JHARKHAND'):            'EAST SINGHBUM',
    ('FIROZPUR',                     'PUNJAB'):               'FEROZEPUR',
    ('GOMTI',                        'TRIPURA'):              'GOMATI',
    ('HAORA',                        'WEST BENGAL'):          'HOWRAH',
    ('HUGLI',                        'WEST BENGAL'):          'HOOGHLY',
    ('JAGATSINGHPUR',                'ODISHA'):               'JAGATSINGHAPUR',
    ('JAGTIAL',                      'TELANGANA'):            'JAGITIAL',
    ('JAHANABAD',                    'BIHAR'):                'JEHANABAD',
    ('JALOR',                        'RAJASTHAN'):            'JALORE',
    ('JAYASHANKAR BHUPALPALLI',      'TELANGANA'):            'JAYASHANKAR BHUPALAPALLY',
    ('JHUNJHUNUN',                   'RAJASTHAN'):            'JHUNJHUNU',
    ('KALLAKKURICHI',                'TAMIL NADU'):           'KALLAKURICHI',
    ('KAMJANG',                      'MANIPUR'):              'KAMJONG',
    ('KAMRUP RURAL',                 'ASSAM'):                'KAMRUP',
    ('KANCHIPURAM',                  'TAMIL NADU'):           'KANCHEEPURAM',
    ('KAWARDHA',                     'CHHATTISGARH'):         'KABEERDHAM',
    ('KEONJHAR',                     'ODISHA'):               'KENDUJHAR',
    ('KHAIRGARH CHHUIKHADAN GANDAI', 'CHHATTISGARH'):         'KHAIRAGARH CHHUIKHADAN GANDAI',
    ('KOCH BIHAR',                   'WEST BENGAL'):          'COOCH BEHAR',
    ('KOL',                          'KARNATAKA'):            'KOLAR',
    ('KOMARRAM BHEEM',               'TELANGANA'):            'KUMURAM BHEEM ASIFABAD',
    ('LAHUL AND SPITI',              'HIMACHAL PRADESH'):     'LAHAUL AND SPITI',
    ('LAKSHADWEEP',                  'LAKSHADWEEP'):          'LAKSHADWEEP DISTRICT',
    ('LEH',                          'LADAKH'):               'LEH LADAKH',
    ('LEPA RADA',                    'ARUNACHAL PRADESH'):    'LEPARADA',
    ('MALDAH',                       'WEST BENGAL'):          'MALDA',
    ('MALER KOTLA',                  'PUNJAB'):               'MALERKOTLA',
    ('MEDCHAL MALKAJGIRI',           'TELANGANA'):            'MEDCHAL MALKAJGIRI',
    ('MEDCHAL_MALKAJGIRI',           'TELANGANA'):            'MEDCHAL MALKAJGIRI',
    ('MOHLA MANPUR AMBAGARH CHOWKI', 'CHHATTISGARH'):         'MOHLA MANPUR AMBAGARH CHOUKI',
    ('MUMBAI',                       'MAHARASHTRA'):          'MUMBAI CITY',
    ('NABARANGAPUR',                 'ODISHA'):               'NABARANGPUR',
    ('NARSINGHPUR',                  'MADHYA PRADESH'):       'NARSIMHAPUR',
    ('NORTH TWENTY FOUR PARGANAS',   'WEST BENGAL'):          'NORTH 24 PARGANAS',
    ('NUAPARHA',                     'ODISHA'):               'NUAPADA',
    ('PAPUMPARE',                    'ARUNACHAL PRADESH'):    'PAPUM PARE',
    ('PASCHIM BARDDHAMAN',           'WEST BENGAL'):          'PASCHIM BARDHAMAN',
    ('PASHCHIMI CHAMPARAN',          'BIHAR'):                'PASHCHIM CHAMPARAN',
    ('PHRBA MEDINIPUR',              'WEST BENGAL'):          'PURBA MEDINIPUR',
    ('PUNCH',                        'JAMMU AND KASHMIR'):    'POONCH',
    ('PURBA BARDDHAMAN',             'WEST BENGAL'):          'PURBA BARDHAMAN',
    ('PURULIYA',                     'WEST BENGAL'):          'PURULIA',
    ('RAJAURI',                      'JAMMU AND KASHMIR'):    'RAJOURI',
    ('RAIGARH',                      'MAHARASHTRA'):          'RAIGAD',
    ('RAYAGARHA',                    'ODISHA'):               'RAYAGADA',
    ('RIASI',                        'JAMMU AND KASHMIR'):    'REASI',
    ('RUDRAPRAYAG',                  'UTTARAKHAND'):          'RUDRA PRAYAG',
    ('SAHIBGANJ',                    'JHARKHAND'):            'SAHEBGANJ',
    ('SAIHA',                        'MIZORAM'):              'SIAHA',
    ('SAS NAGAR',                    'PUNJAB'):               'S.A.S NAGAR',
    ('SHAHADARA',                    'DELHI'):                'SHAHDARA',
    ('SHUPIYAN',                     'JAMMU AND KASHMIR'):    'SHOPIAN',
    ('SIBSAGAR',                     'ASSAM'):                'SIVASAGAR',
    ('SOUTH 24PARGANAS',             'WEST BENGAL'):          'SOUTH 24 PARGANAS',
    ('SUBARNAPUR',                   'ODISHA'):               'SONEPUR',
    ('SURIYAPET',                    'TELANGANA'):            'SURYAPET',
    ('THOOTHUKUDI',                  'TAMIL NADU'):           'THOOTHUKKUDI',
    ('UDHAM SINGH NAGAR',            'UTTARAKHAND'):          'UDAM SINGH NAGAR',
    ('UTTARKASHI',                   'UTTARAKHAND'):          'UTTAR KASHI',
    ('VIJAYANAGARA',                 'KARNATAKA'):            'VIJAYANAGAR',
    ('WANPARTI',                     'TELANGANA'):            'WANAPARTHY',
    ('YSR KADAPA',                   'ANDHRA PRADESH'):       'Y.S.R.',
    # Karnataka truncated names
    ('BALL',        'KARNATAKA'): 'BALLARI',
    ('BELAG',       'KARNATAKA'): 'BELAGAVI',
    ('CHIKKABALL',  'KARNATAKA'): 'CHIKKABALLAPURA',
    # Unmappable: Pakistan-administered, new districts, special cases
    ('BICHOM',       'ARUNACHAL PRADESH'):                       None,
    ('KEYI PANYOR',  'ARUNACHAL PRADESH'):                       None,
    ('MIRPUR',       'JAMMU AND KASHMIR'):                       None,
    ('MUZAFFARABAD', 'JAMMU AND KASHMIR'):                       None,
    ('NAZUL',        'DELHI'):                                   None,
    ('ISLAND',       'DADRA AND NAGAR HAVELI AND DAMAN AND DIU'): None,
    # Single-letter Karnataka artifacts from encoding corruption
    ('B',  'KARNATAKA'): None, ('CH', 'KARNATAKA'): None,
    ('D',  'KARNATAKA'): None, ('DH', 'KARNATAKA'): None,
    ('H',  'KARNATAKA'): None, ('R',  'KARNATAKA'): None,
    ('Y',  'KARNATAKA'): None,
}

def get_district_norm(row):
    """Unified: fix encoding, handle special cases, apply manual map."""
    orig = fix_encoding(str(row.get('district', '')).upper().strip())
    # Handle Bengaluru before stripping parentheses
    if 'BENGAL' in orig and '(RURAL)' in orig:
        norm = 'BENGALURU RURAL'
    elif 'BENGAL' in orig and '(URBAN)' in orig:
        norm = 'BENGALURU URBAN'
    else:
        norm = normalise_name(orig)
    key = (norm, row.get('state_norm', ''))
    if key in DISTRICT_MANUAL_MAP:
        mapped = DISTRICT_MANUAL_MAP[key]
        return mapped if mapped is not None else norm
    return norm

print("Functions defined")

## 6. Apply cleaning and build join keys

Normalise names on both sides, align state names, build `district_norm||state_norm` join keys.

In [ ]:
# Clean GeoJSON
gdf['district_clean'] = gdf['district'].apply(fix_encoding)
gdf['state_clean']    = gdf['state'].apply(fix_encoding)
gdf['state_norm']     = gdf['state_clean'].apply(normalise_state).apply(lambda x: STATE_MAP.get(x, x))

# Clean LGD master
master['district_norm'] = master['district_name_english'].apply(normalise_name)
master['state_norm']    = master['state_name_english'].apply(normalise_state).apply(lambda x: STATE_MAP.get(x, x))

# Verify state alignment
geo_states = set(s for s in gdf['state_norm'].unique() if isinstance(s, str))
lgd_states = set(s for s in master['state_norm'].drop_duplicates().values if isinstance(s, str))
print(f"State mismatches GeoJSON vs LGD: {sorted(geo_states - lgd_states)}")
print(f"State mismatches LGD vs GeoJSON: {sorted(lgd_states - geo_states)}")

# Apply district norm and build join keys
gdf['district_norm'] = gdf.apply(get_district_norm, axis=1)
gdf['join_key'] = gdf.apply(
    lambda r: f"{r['district_norm']}||{r['state_norm']}"
    if pd.notna(r['district_norm']) and pd.notna(r['state_norm']) else None,
    axis=1
)
master['join_key'] = master['district_norm'] + '||' + master['state_norm']

geo_keys = set(k for k in gdf['join_key'].unique() if isinstance(k, str) and '||' in k)
lgd_keys = set(k for k in master.drop_duplicates('district_code')['join_key'].unique() if isinstance(k, str) and '||' in k)
print(f"\nInitial matches:   {len(geo_keys & lgd_keys)}")
print(f"Unmatched GeoJSON: {len(geo_keys - lgd_keys)}")
print(f"Unmatched LGD:     {len(lgd_keys - geo_keys)}")

## 7. Add missing districts to LGD master

Kolkata (315) and Mumbai City (482) are absent from the data.gov.in CSV but confirmed in LGD.
These are municipal corporation districts excluded from the revenue district download.

In [ ]:
missing_districts = pd.DataFrame([
    {
        'state_code': 19, 'state_name_english': 'West Bengal',
        'state_census2011_code': 19.0, 'district_code': 315,
        'district_name_english': 'Kolkata', 'district_census2011_code': 315.0,
        'state_or_ut': 'S', 'district_norm': 'KOLKATA',
        'state_norm': 'WEST BENGAL', 'join_key': 'KOLKATA||WEST BENGAL'
    },
    {
        'state_code': 27, 'state_name_english': 'Maharashtra',
        'state_census2011_code': 27.0, 'district_code': 482,
        'district_name_english': 'Mumbai City', 'district_census2011_code': 482.0,
        'state_or_ut': 'S', 'district_norm': 'MUMBAI CITY',
        'state_norm': 'MAHARASHTRA', 'join_key': 'MUMBAI CITY||MAHARASHTRA'
    }
])

master = pd.concat([master, missing_districts], ignore_index=True)
print(f"Master districts after addition: {master['district_code'].nunique()}")

## 8. Reconstruct missing Karnataka districts

The district GeoJSON uses older Karnataka boundaries and is missing 9 districts.
We dissolve sub-district polygons (which have correct LGD codes) to reconstruct them.

In [ ]:
MISSING_KAR = ['BAGALKOTE', 'BENGALURU RURAL', 'CHAMARAJANAGARA',
               'DAVANGERE', 'DHARWAD', 'HASSAN', 'HAVERI', 'RAMANAGARA', 'YADGIR']

missing_kar_lgd = master[
    (master['state_norm'] == 'KARNATAKA') &
    (master['district_norm'].isin(MISSING_KAR))
][['district_code', 'district_name_english', 'district_norm']].drop_duplicates()

print(f"Missing Karnataka districts: {len(missing_kar_lgd)}")

# Dissolve sub-districts by LGD district code
missing_kar_codes = missing_kar_lgd['district_code'].tolist()
kar_dissolved = (
    gdf_sub[gdf_sub['Dist_LGD'].isin(missing_kar_codes)]
    .dissolve(by='Dist_LGD')
    .reset_index()
)
kar_dissolved['Dist_LGD'] = kar_dissolved['Dist_LGD'].astype(int)
print(f"Dissolved: {len(kar_dissolved)} districts")

# Build rows matching gdf structure
kar_rows = kar_dissolved.merge(missing_kar_lgd, left_on='Dist_LGD', right_on='district_code', how='left')
kar_rows['district']       = kar_rows['dtname'].str.upper()
kar_rows['district_clean'] = kar_rows['dtname']
kar_rows['state']          = 'KARNATAKA'
kar_rows['state_clean']    = 'KARNATAKA'
kar_rows['state_norm']     = 'KARNATAKA'
kar_rows['join_key']       = kar_rows['district_norm'] + '||KARNATAKA'
kar_rows = kar_rows[['district', 'district_clean', 'state', 'state_clean',
                      'state_norm', 'district_norm', 'join_key', 'geometry']]

# Append, drop nulls, deduplicate (keep dissolved over corrupt original)
gdf = pd.concat([gdf, kar_rows], ignore_index=True)
gdf = gdf[gdf['state_norm'].notna()].copy()
gdf = gdf.drop_duplicates(subset='join_key', keep='last')

print(f"GeoJSON rows after Karnataka fix: {len(gdf)}")
print(f"Karnataka districts: {gdf[gdf['state_norm'] == 'KARNATAKA']['district_norm'].nunique()}")

## 9. Final match check

Verify match counts. Expected: ~783 matches, minimal unmatched.

In [ ]:
geo_keys = set(k for k in gdf['join_key'].unique() if isinstance(k, str) and '||' in k)
lgd_keys = set(k for k in master.drop_duplicates('district_code')['join_key'].unique() if isinstance(k, str) and '||' in k)

print(f"Matches:           {len(geo_keys & lgd_keys)}")
print(f"Unmatched GeoJSON: {len(geo_keys - lgd_keys)}")
print(f"Unmatched LGD:     {len(lgd_keys - geo_keys)}")

unmatched_geo = sorted([k for k in geo_keys - lgd_keys if isinstance(k, str) and '||' in k])
unmatched_lgd = sorted([k for k in lgd_keys - geo_keys if isinstance(k, str) and '||' in k])

if unmatched_geo:
    print("\nUnmatched GeoJSON:")
    for k in unmatched_geo:
        d, s = k.split('||')
        print(f"  {d} | {s}")

if unmatched_lgd:
    print("\nUnmatched LGD (recently created districts not yet in GeoJSON):")
    for k in unmatched_lgd:
        d, s = k.split('||')
        print(f"  {d} | {s}")

## 10. Final join and save

In [ ]:
lgd_districts_unique = master[[
    'district_code', 'district_name_english',
    'state_code', 'state_name_english',
    'state_census2011_code', 'district_census2011_code',
    'state_or_ut', 'join_key'
]].drop_duplicates('district_code')

gdf_joined = gdf.merge(lgd_districts_unique, on='join_key', how='left')

# Validate
print(f"GeoJSON rows:          {len(gdf)}")
print(f"Joined rows:           {len(gdf_joined)}")
print(f"Matched with LGD code: {gdf_joined['district_code'].notna().sum()}")
print(f"Unmatched:             {gdf_joined['district_code'].isna().sum()}")
print(f"Duplicate rows:        {gdf_joined.duplicated('join_key').sum()}")

# State misalignment check — catches any cross-state false matches
matched = gdf_joined[gdf_joined['district_code'].notna()].copy()
matched['_state_lgd'] = matched['state_name_english'].str.upper().str.strip().str.replace('THE ', '', regex=False)
matched['_state_geo'] = matched['state_norm'].str.replace('THE ', '', regex=False)
misaligned = matched[matched['_state_geo'] != matched['_state_lgd']]
print(f"\nState misalignments: {len(misaligned)}")
if len(misaligned) > 0:
    print(misaligned[['district_clean', 'district_name_english', '_state_geo', '_state_lgd']].to_string())

# Save
gdf_joined.to_file(PROCESSED / "india_districts_with_lgd.geojson", driver='GeoJSON')
print(f"\nSaved: india_districts_with_lgd.geojson")
print(f"Coverage: {gdf_joined['district_code'].notna().sum() / len(gdf_joined) * 100:.1f}%")